# Healthcare Data Quality Assurance and SQL Analytics

This notebook demonstrates how to load refined healthcare data, perform extensive data quality checks using PySpark, store the results into a Gold-layer quality reporting table, and execute SQL analytical queries to generate operational healthcare insights.

### Step 1: Load Silver Hospital Records
We start by loading the dataset `silver_hospital_records` from the `workspace.healthcare` database into a PySpark DataFrame and displaying its contents to inspect initial records.

In [ ]:
hospital_records = spark.table("workspace.healthcare.silver_hospital_records")

display(hospital_records)

admission_id,patient_id,first_name,last_name,gender,age,city,doctor_id,doctor_name,department,specialization,diagnosis,admission_date,discharge_date,length_of_stay
A004,P004,Sneha,Patel,FEMALE,31,Bangalore,D004,Dr. Sneha Reddy,Pediatrics,Pediatrician,Viral Fever,2026-02-08,2026-02-12,4
A003,P003,Arjun,Reddy,MALE,48,Hyderabad,D003,Dr. Amit Verma,Orthopedics,Orthopedic Surgeon,Bone Fracture,2026-02-05,2026-02-10,5
A005,P005,Vikram,Singh,MALE,44,Delhi,D005,Dr. Arjun Rao,General Medicine,General Physician,Diabetes,2026-03-01,2026-03-04,3
A001,P001,Rahul,Sharma,MALE,41,Hyderabad,D001,Dr. Rajesh Kumar,Cardiology,Cardiologist,Heart Disease,2026-01-10,2026-01-15,5
A002,P002,Priya,Rao,FEMALE,36,Mumbai,D002,Dr. Priya Sharma,Neurology,Neurologist,Migraine,2026-01-12,2026-01-18,6


### Step 2: Inspect Schema
We check column names, data types, and nullability characteristics using `printSchema()`.

In [ ]:
hospital_records.printSchema()

root
 |-- admission_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- specialization: string (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- admission_date: date (nullable = true)
 |-- discharge_date: date (nullable = true)
 |-- length_of_stay: integer (nullable = true)



### Step 3: Check Null Counts across Columns
We construct a dynamic programmatic aggregation to count the number of NULL values for every column in the dataset.

In [ ]:
from pyspark.sql.functions import (
    col,
    sum,
    when
)

null_check = hospital_records.select(
    [
        sum(
            when(col(column_name).isNull(), 1)
            .otherwise(0)
        ).alias(column_name)
        
        for column_name in hospital_records.columns
    ]
)

display(null_check)

admission_id,patient_id,first_name,last_name,gender,age,city,doctor_id,doctor_name,department,specialization,diagnosis,admission_date,discharge_date,length_of_stay
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### Step 4: Duplicate Primary Key Check
We group by `admission_id` and filter for counts greater than 1 to ensure primary key uniqueness.

In [ ]:
duplicate_admissions = hospital_records.groupBy(
    "admission_id"
).count().filter(
    col("count") > 1
)

display(duplicate_admissions)

admission_id,count


### Step 5: Check Range Validity for Age
Validate that patient ages fall within valid biological boundaries (between 0 and 120).

In [ ]:
invalid_age_records = hospital_records.filter(
    (col("age") < 0) |
    (col("age") > 120)
)

display(invalid_age_records)

admission_id,patient_id,first_name,last_name,gender,age,city,doctor_id,doctor_name,department,specialization,diagnosis,admission_date,discharge_date,length_of_stay


### Step 6: Identify Records Missing Critical Data
Check for missing mandatory values in essential fields such as `doctor_name`.

In [ ]:
missing_doctor_records = hospital_records.filter(
    col("doctor_name").isNull()
)

display(missing_doctor_records)

admission_id,patient_id,first_name,last_name,gender,age,city,doctor_id,doctor_name,department,specialization,diagnosis,admission_date,discharge_date,length_of_stay


### Step 7: Imports for Timestamp functions
Import necessary PySpark SQL functions (`lit`, `current_timestamp`) for generating report metadata.

In [ ]:
from pyspark.sql.functions import (
    lit,
    current_timestamp
)

### Step 8: Calculate Violation Metrics
Filter for negative length of stay and compute total counts across all quality check conditions.

In [ ]:
# Define invalid_stay_records
invalid_stay_records = hospital_records.filter(
    (col("length_of_stay") < 0)
)

total_records = hospital_records.count()

duplicate_count = duplicate_admissions.count()

invalid_age_count = invalid_age_records.count()

invalid_stay_count = invalid_stay_records.count()

missing_doctor_count = missing_doctor_records.count()

### Step 9: Construct Data Quality Audit Structure
Build a list of `Row` objects evaluating PASS/FAIL statuses based on metric threshold results.

In [ ]:
from pyspark.sql import Row

quality_data = [
    
    Row(
        check_name="Total Records",
        invalid_records=0,
        total_records=total_records,
        status="INFO"
    ),
    
    Row(
        check_name="Duplicate Admissions",
        invalid_records=duplicate_count,
        total_records=total_records,
        status=(
            "PASS"
            if duplicate_count == 0
            else "FAIL"
        )
    ),
    
    Row(
        check_name="Invalid Age",
        invalid_records=invalid_age_count,
        total_records=total_records,
        status=(
            "PASS"
            if invalid_age_count == 0
            else "FAIL"
        )
    ),
    
    Row(
        check_name="Invalid Length of Stay",
        invalid_records=invalid_stay_count,
        total_records=total_records,
        status=(
            "PASS"
            if invalid_stay_count == 0
            else "FAIL"
        )
    ),
    
    Row(
        check_name="Missing Doctor Information",
        invalid_records=missing_doctor_count,
        total_records=total_records,
        status=(
            "PASS"
            if missing_doctor_count == 0
            else "FAIL"
        )
    )
]

### Step 10: Convert Audit Records to Spark DataFrame
Transform the row objects into a structured Spark DataFrame.

In [ ]:
quality_report = spark.createDataFrame(
    quality_data
)

### Step 11: Add Audit Timestamp
Append an execution timestamp column (`check_timestamp`) to trace when quality checks occurred.

In [ ]:
quality_report = quality_report.withColumn(
    "check_timestamp",
    current_timestamp()
)

display(quality_report)

check_name,invalid_records,total_records,status,check_timestamp
Total Records,0,5,INFO,2026-09-04T07:17:12.449Z
Duplicate Admissions,0,5,PASS,2026-09-04T07:17:12.449Z
Invalid Age,0,5,PASS,2026-09-04T07:17:12.449Z
Invalid Length of Stay,0,5,PASS,2026-09-04T07:17:12.449Z
Missing Doctor Information,0,5,PASS,2026-09-04T07:17:12.449Z


### Step 12: Write Gold Quality Report Table
Persist the compiled quality audit results to the Gold Delta table `gold_data_quality_report`.

In [ ]:
quality_report.write.mode("overwrite").saveAsTable(
    "workspace.healthcare.gold_data_quality_report"
)

display(
    spark.table(
        "workspace.healthcare.gold_data_quality_report"
    )
)

check_name,invalid_records,total_records,status,check_timestamp
Total Records,0,5,INFO,2026-09-04T07:22:28.929Z
Duplicate Admissions,0,5,PASS,2026-09-04T07:22:28.929Z
Invalid Age,0,5,PASS,2026-09-04T07:22:28.929Z
Invalid Length of Stay,0,5,PASS,2026-09-04T07:22:28.929Z
Missing Doctor Information,0,5,PASS,2026-09-04T07:22:28.929Z


### Step 13: Create Temporary View for Spark SQL Operations
Register `hospital_records` as a local temporary view named `hospital_records` to query it via standard Spark SQL syntax.

In [ ]:
hospital_records.createOrReplaceTempView(
    "hospital_records"
)

### Step 14: SQL Initialization
Cell initialization placeholder for switching execution modes.

In [ ]:
%sql

### Step 15: Retrieve All Records via SQL
Query the temporary view directly using `%sql` cell magic to verify SQL accessibility.

In [ ]:
%sql

SELECT *
FROM hospital_records;

admission_id,patient_id,first_name,last_name,gender,age,city,doctor_id,doctor_name,department,specialization,diagnosis,admission_date,discharge_date,length_of_stay
A004,P004,Sneha,Patel,FEMALE,31,Bangalore,D004,Dr. Sneha Reddy,Pediatrics,Pediatrician,Viral Fever,2026-02-08,2026-02-12,4
A003,P003,Arjun,Reddy,MALE,48,Hyderabad,D003,Dr. Amit Verma,Orthopedics,Orthopedic Surgeon,Bone Fracture,2026-02-05,2026-02-10,5
A005,P005,Vikram,Singh,MALE,44,Delhi,D005,Dr. Arjun Rao,General Medicine,General Physician,Diabetes,2026-03-01,2026-03-04,3
A001,P001,Rahul,Sharma,MALE,41,Hyderabad,D001,Dr. Rajesh Kumar,Cardiology,Cardiologist,Heart Disease,2026-01-10,2026-01-15,5
A002,P002,Priya,Rao,FEMALE,36,Mumbai,D002,Dr. Priya Sharma,Neurology,Neurologist,Migraine,2026-01-12,2026-01-18,6


### Step 16: Patient Distribution by Department
Aggregate distinct patient counts per department to analyze patient load distribution.

In [ ]:
%sql

SELECT
    department,
    COUNT(DISTINCT patient_id) AS total_patients
FROM hospital_records
GROUP BY department
ORDER BY total_patients DESC;

department,total_patients
Cardiology,1
Orthopedics,1
Pediatrics,1
General Medicine,1
Neurology,1


### Step 17: Average Length of Stay by Department
Calculate rounded average length of stay (in days) grouped by medical department.

In [ ]:
%sql

SELECT
    department,
    ROUND(
        AVG(length_of_stay),
        2
    ) AS average_length_of_stay
FROM hospital_records
GROUP BY department
ORDER BY average_length_of_stay DESC;

department,average_length_of_stay
Neurology,6.0
Cardiology,5.0
Orthopedics,5.0
Pediatrics,4.0
General Medicine,3.0


### Step 18: Total Admissions by Diagnosis
Summarize total patient admissions grouped by primary diagnosis.

In [ ]:
%sql

SELECT
    diagnosis,
    COUNT(*) AS total_admissions
FROM hospital_records
GROUP BY diagnosis
ORDER BY total_admissions DESC;

diagnosis,total_admissions
Heart Disease,1
Bone Fracture,1
Viral Fever,1
Diabetes,1
Migraine,1


### Step 19: Doctor Performance Analysis
Evaluate total admissions and distinct patients handled per physician across departments.

In [ ]:
%sql

SELECT
    doctor_name,
    department,
    COUNT(DISTINCT admission_id) AS total_admissions,
    COUNT(DISTINCT patient_id) AS total_patients
FROM hospital_records
GROUP BY
    doctor_name,
    department
ORDER BY total_admissions DESC;

doctor_name,department,total_admissions,total_patients
Dr. Rajesh Kumar,Cardiology,1,1
Dr. Amit Verma,Orthopedics,1,1
Dr. Sneha Reddy,Pediatrics,1,1
Dr. Arjun Rao,General Medicine,1,1
Dr. Priya Sharma,Neurology,1,1


### Step 20: Geographic Patient Demographics
Analyze geographical breakdown of patients grouped by city of origin.

In [ ]:
%sql

SELECT
    city,
    COUNT(DISTINCT patient_id) AS total_patients
FROM hospital_records
GROUP BY city
ORDER BY total_patients DESC;

city,total_patients
Hyderabad,2
Bangalore,1
Delhi,1
Mumbai,1


### Step 21: Create Master Healthcare Analytics View
Create or replace a global analytics SQL View named `healthcare_analytics_view` over the silver record source.

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW
workspace.healthcare.healthcare_analytics_view
AS
SELECT
    admission_id,
    patient_id,
    first_name,
    last_name,
    age,
    gender,
    city,
    doctor_id,
    doctor_name,
    department,
    specialization,
    diagnosis,
    admission_date,
    discharge_date,
    length_of_stay
FROM workspace.healthcare.silver_hospital_records
""")

DataFrame[]

### Step 22: Validate Created Analytics View
Display contents from the newly created `workspace.healthcare.healthcare_analytics_view`.

In [ ]:
display(
    spark.table(
        "workspace.healthcare.healthcare_analytics_view"
    )
)

admission_id,patient_id,first_name,last_name,age,gender,city,doctor_id,doctor_name,department,specialization,diagnosis,admission_date,discharge_date,length_of_stay
A004,P004,Sneha,Patel,31,FEMALE,Bangalore,D004,Dr. Sneha Reddy,Pediatrics,Pediatrician,Viral Fever,2026-02-08,2026-02-12,4
A003,P003,Arjun,Reddy,48,MALE,Hyderabad,D003,Dr. Amit Verma,Orthopedics,Orthopedic Surgeon,Bone Fracture,2026-02-05,2026-02-10,5
A005,P005,Vikram,Singh,MALE,44,Delhi,D005,Dr. Arjun Rao,General Medicine,General Physician,Diabetes,2026-03-01,2026-03-04,3
A001,P001,Rahul,Sharma,MALE,41,Hyderabad,D001,Dr. Rajesh Kumar,Cardiology,Cardiologist,Heart Disease,2026-01-10,2026-01-15,5
A002,P002,Priya,Rao,36,FEMALE,Mumbai,D002,Dr. Priya Sharma,Neurology,Neurologist,Migraine,2026-01-12,2026-01-18,6


### Step 23: List Workspace Schema Tables and Views
Display all registered tables and views inside the `workspace.healthcare` schema.

In [ ]:
spark.sql("""
SHOW TABLES IN workspace.healthcare
""").show(truncate=False)

+----------+---------------------------+-----------+
|database  |tableName                  |isTemporary|
+----------+---------------------------+-----------+
|healthcare|bronze_admissions          |false      |
|healthcare|bronze_doctors             |false      |
|healthcare|bronze_patients            |false      |
|healthcare|gold_data_quality_report   |false      |
|healthcare|gold_department_performance|false      |
|healthcare|gold_disease_summary       |false      |
|healthcare|gold_doctor_performance    |false      |
|healthcare|gold_hospital_kpis         |false      |
|healthcare|gold_patient_summary       |false      |
|healthcare|healthcare_analytics_view  |false      |
|healthcare|silver_admissions          |false      |
|healthcare|silver_doctors             |false      |
|healthcare|silver_hospital_records    |false      |
|healthcare|silver_patients            |false      |
|          |hospital_records           |true       |
+----------+---------------------------+------